In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')  # ensure UTC+8 timezone is available for any analysis steps

# Load files (reuse variable name df to be memory-efficient, then keep summaries)
df = pd.read_csv('metric.csv')
metric_rows = df.shape[0]
# metric: group by cmdb_id and kpi_name
metric_summary = (
    df.groupby(['cmdb_id', 'kpi_name'])
      .size()
      .reset_index(name='count_of_points')
)
metric_summary['source_file'] = 'metric'
metric_top50 = metric_summary.sort_values('count_of_points', ascending=False).head(50)[
    ['source_file', 'cmdb_id', 'kpi_name', 'count_of_points']
]

# trace
df = pd.read_csv('trace.csv')
trace_rows = df.shape[0]
# rename trace_name -> kpi_name for generic handling
df = df.rename(columns={'trace_name': 'kpi_name'})
trace_summary = (
    df.groupby(['cmdb_id', 'kpi_name'])
      .size()
      .reset_index(name='count_of_points')
)
trace_summary['source_file'] = 'trace'
trace_top50 = trace_summary.sort_values('count_of_points', ascending=False).head(50)[
    ['source_file', 'cmdb_id', 'kpi_name', 'count_of_points']
]

# log
df = pd.read_csv('log.csv')
log_rows = df.shape[0]
df = df.rename(columns={'log_name': 'kpi_name'})
log_summary = (
    df.groupby(['cmdb_id', 'kpi_name'])
      .size()
      .reset_index(name='count_of_points')
)
log_summary['source_file'] = 'log'
log_top50 = log_summary.sort_values('count_of_points', ascending=False).head(50)[
    ['source_file', 'cmdb_id', 'kpi_name', 'count_of_points']
]

# Totals compact summary
totals = pd.DataFrame({
    'source_file': ['metric', 'trace', 'log'],
    'total_rows': [metric_rows, trace_rows, log_rows]
})

# Display compact results (variables returned)
metric_top50, trace_top50, log_top50, totals

```
Out[1]:
```
```python
# Summarize previous aggregation results (variables from prior execution are reused)
import pandas as pd

# Safely access existing summary variables
_metric = metric_top50 if 'metric_top50' in globals() else pd.DataFrame()
_trace = trace_top50 if 'trace_top50' in globals() else pd.DataFrame()
_log = log_top50 if 'log_top50' in globals() else pd.DataFrame()
_totals = totals if 'totals' in globals() else pd.DataFrame(columns=['source_file','total_rows']).set_index('source_file')

def top_examples(df, n=3):
    rows = []
    for _, r in df.head(n).iterrows():
        rows.append(f"{r['cmdb_id']} | {r['kpi_name']} ({int(r['count_of_points'])})")
    return rows if rows else ["(no data)"]

metric_examples = top_examples(_metric, 3)
trace_examples = top_examples(_trace, 3)
log_examples = top_examples(_log, 3)

metric_total = int(_totals.loc['metric','total_rows']) if 'metric' in _totals.index else None
trace_total = int(_totals.loc['trace','total_rows']) if 'trace' in _totals.index else None
log_total = int(_totals.loc['log','total_rows']) if 'log' in _totals.index else None

summary = (
    f"Summary of telemetry files:\n"
    f"- Totals: metric={metric_total:,} rows; trace={trace_total:,} rows; log={log_total:,} rows.\n\n"
    f"- Top metric cmdb_id+kpi examples (by point count):\n  1) {metric_examples[0]}\n  2) {metric_examples[1]}\n  3) {metric_examples[2]}\n\n"
    f"- Top trace cmdb_id+kpi examples (by point count):\n  1) {trace_examples[0]}\n  2) {trace_examples[1]}\n  3) {trace_examples[2]}\n\n"
    f"- Top log cmdb_id+kpi examples (by point count):\n  1) {log_examples[0]}\n  2) {log_examples[1]}\n  3) {log_examples[2]}\n\n"
    f"Notes: Each returned list contains up to the top 50 (we show top 3 examples here). Counts are the number of data points per (cmdb_id, kpi_name) combination."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     source_file             cmdb_id                                           kpi_name  count_of_points
5345      metric  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....               61
0         metric           adservice                                     app.grpc.count               61
5306      metric  shippingservice2-0  container.node-6.container_network_transmit_MB...               61
5307      metric  shippingservice2-0  container.node-6.container_network_transmit_er...               61
5308      metric  shippingservice2-0  container.node-6.container_network_transmit_pa...               61
...          ...                 ...                                                ...              ...
53        metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               61
54        metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               61
55        metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               61
56        metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               61
57        metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               61

[50 rows x 4 columns],     source_file      cmdb_id                                        kpi_name  count_of_points
850       trace  frontend2-0                           trace.self.error_rate               31
849       trace  frontend2-0                         trace.self.duration_p95               31
848       trace  frontend2-0                        trace.self.duration_mean               31
851       trace  frontend2-0                            trace.self.row_count               31
660       trace   frontend-1                   trace.from_root.duration_mean               30
..          ...          ...                                             ...              ...
633       trace   frontend-0   trace.to_productcatalogservice-2.duration_p95               30
634       trace   frontend-0     trace.to_productcatalogservice-2.error_rate               30
635       trace   frontend-0      trace.to_productcatalogservice-2.row_count               30
636       trace   frontend-0  trace.to_recommendationservice-0.duration_mean               30
637       trace   frontend-0   trace.to_recommendationservice-0.duration_p95               30

[50 rows x 4 columns],    source_file                  cmdb_id         kpi_name  count_of_points
3          log              adservice-1    log.row_count               31
2          log              adservice-1  log.error_count               31
9          log            cartservice-1    log.row_count               31
8          log            cartservice-1  log.error_count               31
7          log            cartservice-0    log.row_count               31
..         ...                      ...              ...              ...
37         log         paymentservice-0    log.row_count               30
45         log  productcatalogservice-1    log.row_count               30
44         log  productcatalogservice-1  log.error_count               30
43         log  productcatalogservice-0    log.row_count               30
42         log  productcatalogservice-0  log.error_count               30

[50 rows x 4 columns],   source_file  total_rows
0      metric      325376
1       trace       36316
2         log        1842)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# ensure UTC+8 timezone is available (per rule) even if not used directly here
tz = pytz.timezone('Asia/Shanghai')

# Load metric.csv and compute aggregations
df_metric = pd.read_csv('metric.csv')
metric_thresholds = (
    df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count_of_points='size',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05),
        mean='mean',
        max='max'
    )
    .reset_index()
)
metric_thresholds['source_file'] = 'metric'
# Order columns
metric_thresholds = metric_thresholds[['source_file', 'cmdb_id', 'kpi_name', 'count_of_points', 'p95', 'p5', 'mean', 'max']]

# Load trace.csv, rename trace_name->kpi_name, compute aggregations
df_trace = pd.read_csv('trace.csv').rename(columns={'trace_name': 'kpi_name'})
trace_thresholds = (
    df_trace.groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count_of_points='size',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05),
        mean='mean',
        max='max'
    )
    .reset_index()
)
trace_thresholds['source_file'] = 'trace'
trace_thresholds = trace_thresholds[['source_file', 'cmdb_id', 'kpi_name', 'count_of_points', 'p95', 'p5', 'mean', 'max']]

# Load log.csv, rename log_name->kpi_name, compute aggregations
df_log = pd.read_csv('log.csv').rename(columns={'log_name': 'kpi_name'})
log_thresholds = (
    df_log.groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count_of_points='size',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05),
        mean='mean',
        max='max'
    )
    .reset_index()
)
log_thresholds['source_file'] = 'log'
log_thresholds = log_thresholds[['source_file', 'cmdb_id', 'kpi_name', 'count_of_points', 'p95', 'p5', 'mean', 'max']]

# Prepare compact top-20 views sorted by count_of_points desc for each file
metric_top20 = metric_thresholds.sort_values('count_of_points', ascending=False).head(20)
trace_top20 = trace_thresholds.sort_values('count_of_points', ascending=False).head(20)
log_top20 = log_thresholds.sort_values('count_of_points', ascending=False).head(20)

# Return the top-20 summaries for each file (variables are stored as required)
metric_thresholds.shape, trace_thresholds.shape, log_thresholds.shape, metric_top20, trace_top20, log_top20

```
Out[2]:
```
```python
# Summarize the previously computed per-(cmdb_id,kpi_name) summaries in plain English
# Reuse existing kernel variables if available; otherwise load minimal data to compute counts.

import pandas as pd

# Helper to safely access or load thresholds
def ensure_thresholds(var_name, filename, name_col):
    if var_name in globals():
        return globals()[var_name]
    # minimal fallback: load file and compute same aggregations as before
    df = pd.read_csv(filename)
    if name_col != 'kpi_name':
        df = df.rename(columns={name_col: 'kpi_name'})
    th = (
        df.groupby(['cmdb_id', 'kpi_name'])['value']
          .agg(count_of_points='size',
               p95=lambda x: x.quantile(0.95),
               p5=lambda x: x.quantile(0.05),
               mean='mean',
               max='max')
          .reset_index()
    )
    th['source_file'] = filename.split('.')[0]
    th = th[['source_file','cmdb_id','kpi_name','count_of_points','p95','p5','mean','max']]
    return th

metric_thresholds = ensure_thresholds('metric_thresholds', 'metric.csv', 'kpi_name')
trace_thresholds  = ensure_thresholds('trace_thresholds',  'trace.csv',  'trace_name')
log_thresholds    = ensure_thresholds('log_thresholds',    'log.csv',    'log_name')

# Prepare compact top examples for each file
def top_n_summary(df, n=2):
    top = df.sort_values('count_of_points', ascending=False).head(n)
    lines = []
    for _, r in top.iterrows():
        # keep the kpi_name compact by truncating if very long
        kpi = (r['kpi_name'][:80] + '...') if len(r['kpi_name'])>83 else r['kpi_name']
        lines.append(f"{r['cmdb_id']} | {kpi} | count={int(r['count_of_points'])} | p95={r['p95']:.6g} | mean={r['mean']:.6g}")
    return lines

m_groups = metric_thresholds.shape[0]
t_groups = trace_thresholds.shape[0]
l_groups = log_thresholds.shape[0]

metric_top_examples = top_n_summary(metric_thresholds, 2)
trace_top_examples  = top_n_summary(trace_thresholds, 2)
log_top_examples    = top_n_summary(log_thresholds, 2)

summary = (
    f"Summary of per-(cmdb_id, kpi) aggregations (full files used):\n\n"
    f"Metric (metric.csv): {m_groups:,} unique (cmdb_id, kpi) groups.\n"
    f" Top examples (sorted by data point count):\n  1) {metric_top_examples[0]}\n  2) {metric_top_examples[1]}\n\n"
    f"Trace (trace.csv): {t_groups:,} unique (cmdb_id, kpi) groups.\n"
    f" Top examples:\n  1) {trace_top_examples[0]}\n  2) {trace_top_examples[1]}\n\n"
    f"Log (log.csv): {l_groups:,} unique (cmdb_id, kpi) groups.\n"
    f" Top examples:\n  1) {log_top_examples[0]}\n  2) {log_top_examples[1]}\n\n"
    f"Notes:\n"
    f" - Aggregations computed per (cmdb_id, kpi_name): count_of_points, p95, p5, mean, max.\n"
    f" - Top groups in metric/trace/log show consistent high counts (e.g., 61 for many metric groups, ~30 for trace and log groups),\n"
    f"   indicating those KPIs have full coverage over the collected time range.\n"
    f" - Many log.error_count series have p95=0 (no errors observed at high percentiles in those groups).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((5346, 8), (1232, 8), (62, 8),      source_file             cmdb_id                                           kpi_name  count_of_points           p95            p5          mean           max
5345      metric  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....               61  2.587000e+03  1.492500e+03  2.094393e+03  2.786000e+03
0         metric           adservice                                     app.grpc.count               61  2.580000e+02  2.010000e+02  2.312787e+02  2.760000e+02
5306      metric  shippingservice2-0  container.node-6.container_network_transmit_MB...               61  6.458540e-01  5.937247e-01  6.169501e-01  6.663318e-01
5307      metric  shippingservice2-0  container.node-6.container_network_transmit_er...               61  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
5308      metric  shippingservice2-0  container.node-6.container_network_transmit_pa...               61  3.080000e+02  2.190000e+02  2.648443e+02  4.270000e+02
5309      metric  shippingservice2-0  container.node-6.container_network_transmit_pa...               61  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
5310      metric  shippingservice2-0                 container.node-6.container_sockets               61  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
5311      metric  shippingservice2-0         container.node-6.container_spec_cpu_period               61  1.000000e+05  1.000000e+05  1.000000e+05  1.000000e+05
5312      metric  shippingservice2-0          container.node-6.container_spec_cpu_quota               61  2.000000e+04  2.000000e+04  2.000000e+04  2.000000e+04
5313      metric  shippingservice2-0         container.node-6.container_spec_cpu_shares               61  1.020000e+02  1.020000e+02  1.020000e+02  1.020000e+02
5298      metric  shippingservice2-0              container.node-6.container_memory_rss               61  1.038746e+07  1.038746e+07  1.038746e+07  1.038746e+07
5299      metric  shippingservice2-0             container.node-6.container_memory_swap               61  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
5300      metric  shippingservice2-0         container.node-6.container_memory_usage_MB               61  4.275781e+01  4.263672e+01  4.270322e+01  4.275781e+01
5301      metric  shippingservice2-0   container.node-6.container_memory_working_set_MB               61  3.340625e+01  3.328516e+01  3.335166e+01  3.340625e+01
5302      metric  shippingservice2-0  container.node-6.container_network_receive_MB....               61  5.443716e-02  3.749037e-02  7.343526e-02  6.296234e-01
5303      metric  shippingservice2-0  container.node-6.container_network_receive_err...               61  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
5304      metric  shippingservice2-0  container.node-6.container_network_receive_pac...               61  4.690000e+02  3.425000e+02  4.116639e+02  6.060000e+02
5305      metric  shippingservice2-0  container.node-6.container_network_receive_pac...               61  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
5290      metric  shippingservice2-0            container.node-6.container_memory_cache               61  2.113536e+07  2.113536e+07  2.113536e+07  2.113536e+07
5291      metric  shippingservice2-0          container.node-6.container_memory_failcnt               61  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00,     source_file      cmdb_id                                    kpi_name  count_of_points          p95           p5         mean          max
850       trace  frontend2-0                       trace.self.error_rate               31     0.000000     0.000000     0.000000     0.000000
849       trace  frontend2-0                     trace.self.duration_p95               31     0.008044     0.007451     0.007532     0.008190
848       trace  frontend2-0                    trace.self.duration_mean               31     0.004971     0.004403     0.004642     0.005065
851       trace  frontend2-0                        trace.self.row_count               31  1378.000000  1157.500000  1228.709677  1421.000000
660       trace   frontend-1               trace.from_root.duration_mean               30     0.053774     0.047013     0.060390     0.373949
658       trace   frontend-0       trace.to_shippingservice-2.error_rate               30     0.000000     0.000000     0.000000     0.000000
657       trace   frontend-0     trace.to_shippingservice-2.duration_p95               30     0.000229     0.000062     0.000090     0.000320
656       trace   frontend-0    trace.to_shippingservice-2.duration_mean               30     0.000096     0.000059     0.000069     0.000155
655       trace   frontend-0        trace.to_shippingservice-1.row_count               30     7.000000     3.000000     5.100000     8.000000
654       trace   frontend-0       trace.to_shippingservice-1.error_rate               30     0.000000     0.000000     0.000000     0.000000
653       trace   frontend-0     trace.to_shippingservice-1.duration_p95               30     0.000107     0.000070     0.000085     0.000285
652       trace   frontend-0    trace.to_shippingservice-1.duration_mean               30     0.000080     0.000068     0.000073     0.000116
651       trace   frontend-0        trace.to_shippingservice-0.row_count               30     7.000000     3.000000     5.066667     8.000000
650       trace   frontend-0       trace.to_shippingservice-0.error_rate               30     0.000000     0.000000     0.000000     0.000000
649       trace   frontend-0     trace.to_shippingservice-0.duration_p95               30     0.000301     0.000071     0.000122     0.000327
647       trace   frontend-0  trace.to_recommendationservice-2.row_count               30    21.550000    15.000000    18.300000    23.000000
659       trace   frontend-0        trace.to_shippingservice-2.row_count               30     7.550000     3.000000     5.100000     8.000000
661       trace   frontend-1                trace.from_root.duration_p95               30     0.091564     0.066455     0.098898     0.835818
662       trace   frontend-1                  trace.from_root.error_rate               30     0.000000     0.000000     0.000000     0.000000
663       trace   frontend-1                   trace.from_root.row_count               30    98.000000    80.450000    90.333333   101.000000,    source_file                  cmdb_id         kpi_name  count_of_points    p95     p5        mean     max
3          log              adservice-1    log.row_count               31  114.0   91.0   99.419355   124.0
2          log              adservice-1  log.error_count               31    0.0    0.0    0.000000     0.0
9          log            cartservice-1    log.row_count               31  924.5  734.5  806.387097  1031.0
8          log            cartservice-1  log.error_count               31    0.0    0.0    0.000000     0.0
7          log            cartservice-0    log.row_count               31  926.0  741.0  802.516129   989.0
6          log            cartservice-0  log.error_count               31    0.0    0.0    0.000000     0.0
10         log            cartservice-2  log.error_count               31    0.0    0.0    0.000000     0.0
11         log            cartservice-2    log.row_count               31  865.0  477.5  772.032258   876.0
22         log        currencyservice-2  log.error_count               31    0.0    0.0    0.000000     0.0
23         log        currencyservice-2    log.row_count               31  801.0  585.5  692.580645   806.0
18         log        currencyservice-0  log.error_count               31    0.0    0.0    0.000000     0.0
19         log        currencyservice-0    log.row_count               31  792.0  594.0  692.322581   800.0
20         log        currencyservice-1  log.error_count               31    0.0    0.0    0.000000     0.0
21         log        currencyservice-1    log.row_count               31  789.0  580.0  692.225806   812.0
49         log  recommendationservice-0    log.row_count               31  207.5  180.0  189.741935   221.0
48         log  recommendationservice-0  log.error_count               31    0.0    0.0    0.000000     0.0
51         log  recommendationservice-1    log.row_count               31  208.0  185.0  189.806452   216.0
50         log  recommendationservice-1  log.error_count               31    0.0    0.0    0.000000     0.0
52         log  recommendationservice-2  log.error_count               31    0.0    0.0    0.000000     0.0
53         log  recommendationservice-2    log.row_count               31  205.5  183.0  189.709677   213.0)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')  # UTC+8

# Reuse existing loaded dataframes if available, otherwise load
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv').rename(columns={'trace_name': 'kpi_name'})
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv').rename(columns={'log_name': 'kpi_name'})

# Ensure threshold tables exist in kernel
if 'metric_thresholds' not in globals():
    raise RuntimeError("metric_thresholds not found in kernel")
if 'trace_thresholds' not in globals():
    raise RuntimeError("trace_thresholds not found in kernel")
if 'log_thresholds' not in globals():
    raise RuntimeError("log_thresholds not found in kernel")

# Compute epoch window for 2022-03-21 14:00:00 to 2022-03-21 14:30:00 in Asia/Shanghai (UTC+8)
start_dt = tz.localize(datetime(2022, 3, 21, 14, 0, 0))
end_dt   = tz.localize(datetime(2022, 3, 21, 14, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# Helper to process one source
def process_window(df_raw, thresholds_df, source_name):
    # Ensure kpi_name exists
    if 'kpi_name' not in df_raw.columns:
        raise RuntimeError(f"{source_name} data missing 'kpi_name' column")
    # Filter by time window (timestamps are seconds)
    df_window = df_raw[(df_raw['timestamp'] >= start_ts) & (df_raw['timestamp'] <= end_ts)].copy()
    # Merge thresholds (we only need p95 and p5 from thresholds)
    thr = thresholds_df[['cmdb_id', 'kpi_name', 'p95', 'p5']].copy()
    merged = df_window.merge(thr, on=['cmdb_id', 'kpi_name'], how='left', validate='m:1')
    # Drop rows without thresholds (cannot evaluate anomaly)
    merged = merged.dropna(subset=['p95', 'p5'])
    # Mark anomaly: value >= p95 OR value <= p5
    merged['anomaly'] = (merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])
    # Convert timestamp to int (should already be)
    merged['timestamp'] = merged['timestamp'].astype(int)
    # Store full merged rows into kernel variable
    var_rows_name = f"{source_name}_window_rows"
    globals()[var_rows_name] = merged

    # Aggregate anomalies per (cmdb_id, kpi_name)
    anom_rows = merged[merged['anomaly']].copy()
    if anom_rows.shape[0] == 0:
        # empty summary
        anoms_summary = pd.DataFrame(columns=[
            'source_file','cmdb_id','kpi_name','anomaly_count',
            'earliest_anomaly_timestamp','latest_anomaly_timestamp',
            'max_value_in_window','min_value_in_window','p95','p5',
            'direction','severity_ratio'
        ])
        globals()[f"{source_name}_anoms_window"] = anoms_summary
        return globals()[var_rows_name], anoms_summary

    agg = anom_rows.groupby(['cmdb_id', 'kpi_name']).agg(
        anomaly_count = ('anomaly', 'sum'),
        earliest_anom_ts = ('timestamp', 'min'),
        latest_anom_ts = ('timestamp', 'max'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
        p95 = ('p95', 'first'),
        p5  = ('p5', 'first')
    ).reset_index()

    # Determine direction and severity_ratio
    def compute_direction_and_severity(row):
        # direction high if max_value_in_window >= p95 else low
        if pd.isna(row['p95']) or pd.isna(row['p5']):
            return pd.Series({'direction': None, 'severity_ratio': None})
        if row['max_value_in_window'] >= row['p95']:
            direction = 'high'
            # avoid division by zero
            severity = None
            if row['p95'] != 0:
                severity = row['max_value_in_window'] / row['p95']
        else:
            direction = 'low'
            severity = None
            if row['min_value_in_window'] != 0:
                severity = row['p5'] / row['min_value_in_window']
        return pd.Series({'direction': direction, 'severity_ratio': severity})

    dir_sev = agg.apply(compute_direction_and_severity, axis=1)
    agg = pd.concat([agg, dir_sev], axis=1)

    # Convert earliest/latest timestamps to Asia/Shanghai human-readable strings
    def fmt_ts(ts):
        return datetime.fromtimestamp(int(ts), tz).strftime('%Y-%m-%d %H:%M:%S')
    agg['earliest_anomaly_timestamp'] = agg['earliest_anom_ts'].apply(fmt_ts)
    agg['latest_anomaly_timestamp']   = agg['latest_anom_ts'].apply(fmt_ts)

    # Add source_file column
    agg['source_file'] = source_name

    # Reorder/rename columns per spec
    anoms_summary = agg[[
        'source_file','cmdb_id','kpi_name','anomaly_count',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp',
        'max_value_in_window','min_value_in_window','direction','severity_ratio'
    ]].copy()

    # Store aggregated summary in kernel variable
    globals()[f"{source_name}_anoms_window"] = anoms_summary
    return globals()[var_rows_name], globals()[f"{source_name}_anoms_window"]

# Process metric, trace, log
metric_window_rows, metric_anoms_window = process_window(df_metric, metric_thresholds, 'metric')
trace_window_rows,  trace_anoms_window  = process_window(df_trace,  trace_thresholds,  'trace')
log_window_rows,    log_anoms_window    = process_window(df_log,    log_thresholds,    'log')

# Prepare compact top-20 aggregated anomaly summaries sorted by anomaly_count desc
metric_anoms_top20 = metric_anoms_window.sort_values('anomaly_count', ascending=False).head(20)[
    ['source_file','cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','direction','severity_ratio']
]
trace_anoms_top20 = trace_anoms_window.sort_values('anomaly_count', ascending=False).head(20)[
    ['source_file','cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','direction','severity_ratio']
]
log_anoms_top20 = log_anoms_window.sort_values('anomaly_count', ascending=False).head(20)[
    ['source_file','cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','direction','severity_ratio']
]

# Return compact results (variables are stored in kernel as requested)
metric_anoms_top20, trace_anoms_top20, log_anoms_top20

```
Out[3]:
```
```python
# Summarize anomaly aggregation results in plain English using existing kernel variables.
# Reuse variables produced earlier: metric_anoms_window, trace_anoms_window, log_anoms_window

def short(s, n=80):
    s = str(s)
    return (s if len(s) <= n else s[:n-3] + '...')

metric_df = globals().get('metric_anoms_window', None)
trace_df  = globals().get('trace_anoms_window', None)
log_df    = globals().get('log_anoms_window', None)

def top_examples(df, n=3):
    if df is None or df.shape[0] == 0:
        return ["(no anomalies)"]
    top = df.sort_values('anomaly_count', ascending=False).head(n)
    lines = []
    for _, r in top.iterrows():
        sev = r['severity_ratio']
        sev_str = f"{sev:.3g}" if pd.notna(sev) else "NaN"
        lines.append(f"{r['cmdb_id']} | {short(r['kpi_name'],80)} | count={int(r['anomaly_count'])} | dir={r['direction']} | sev={sev_str}")
    return lines

import pandas as pd
m_cnt = 0 if metric_df is None else metric_df.shape[0]
t_cnt = 0 if trace_df  is None else trace_df.shape[0]
l_cnt = 0 if log_df    is None else log_df.shape[0]

m_top = top_examples(metric_df, 3)
t_top = top_examples(trace_df, 3)
l_top = top_examples(log_df, 3)

summary = (
    "Anomaly summary for window 2022-03-21 14:00:00 to 2022-03-21 14:30:00 (UTC+8):\n\n"
    f"- Metric: {m_cnt} (cmdb_id, kpi) groups flagged as anomalous.\n"
    f"  Top examples:\n"
    f"   1) {m_top[0]}\n"
    f"   2) {m_top[1] if len(m_top)>1 else ''}\n"
    f"   3) {m_top[2] if len(m_top)>2 else ''}\n\n"
    f"- Trace: {t_cnt} (cmdb_id, kpi) groups flagged as anomalous.\n"
    f"  Top examples:\n"
    f"   1) {t_top[0]}\n"
    f"   2) {t_top[1] if len(t_top)>1 else ''}\n"
    f"   3) {t_top[2] if len(t_top)>2 else ''}\n\n"
    f"- Log: {l_cnt} (cmdb_id, kpi) groups flagged as anomalous.\n"
    f"  Top examples:\n"
    f"   1) {l_top[0]}\n"
    f"   2) {l_top[1] if len(l_top)>1 else ''}\n"
    f"   3) {l_top[2] if len(l_top)>2 else ''}\n\n"
    "Key observations:\n"
    "- Many top anomaly groups have anomaly_count = 30 or 31 (i.e., anomalies present throughout the 30-min window),\n"
    "  and their earliest anomaly timestamps are at 2022-03-21 14:00:00 (UTC+8).\n"
    "- Direction is predominantly 'high' (values >= p95) for the reported top groups.\n"
    "- severity_ratio is often NaN for many groups because the p95 or p5 thresholds used for the ratio are zero or\n"
    "  resulted in an undefined division; where numeric, severity_ratio values of ~1.0 indicate values reached the historical p95.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     source_file             cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp direction  severity_ratio
5329      metric  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...             31        2022-03-21 14:00:00      high             NaN
5327      metric  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...             31        2022-03-21 14:00:00      high             NaN
5324      metric  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...             31        2022-03-21 14:00:00      high             NaN
2         metric           adservice                                        app.grpc.rr             31        2022-03-21 14:00:00      high             1.0
3         metric           adservice                                        app.grpc.sr             31        2022-03-21 14:00:00      high             1.0
4         metric           adservice                                     app.http.count             31        2022-03-21 14:00:00      high             1.0
5322      metric  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...             31        2022-03-21 14:00:00      high             NaN
6         metric           adservice                                        app.http.rr             31        2022-03-21 14:00:00      high             1.0
7         metric           adservice                                        app.http.sr             31        2022-03-21 14:00:00      high             1.0
8         metric           adservice    runtime.java_lang_ClassLoading_LoadedClassCount             31        2022-03-21 14:00:00      high             1.0
9         metric           adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...             31        2022-03-21 14:00:00      high             1.0
5283      metric  shippingservice2-0  container.node-6.container_memory_failures.con...             31        2022-03-21 14:00:00      high             NaN
5284      metric  shippingservice2-0  container.node-6.container_memory_failures.hie...             31        2022-03-21 14:00:00      high             NaN
5285      metric  shippingservice2-0  container.node-6.container_memory_failures.hie...             31        2022-03-21 14:00:00      high             NaN
5286      metric  shippingservice2-0      container.node-6.container_memory_mapped_file             31        2022-03-21 14:00:00      high             1.0
5287      metric  shippingservice2-0     container.node-6.container_memory_max_usage_MB             31        2022-03-21 14:00:00      high             1.0
5288      metric  shippingservice2-0              container.node-6.container_memory_rss             31        2022-03-21 14:00:00      high             1.0
5289      metric  shippingservice2-0             container.node-6.container_memory_swap             31        2022-03-21 14:00:00      high             NaN
5275      metric  shippingservice2-0      container.node-6.container_fs_writes./dev/vda             31        2022-03-21 14:00:00      high             NaN
5276      metric  shippingservice2-0     container.node-6.container_fs_writes./dev/vda1             31        2022-03-21 14:00:00      high             NaN,      source_file                  cmdb_id                                        kpi_name  anomaly_count earliest_anomaly_timestamp direction  severity_ratio
850        trace              frontend2-0                           trace.self.error_rate             31        2022-03-21 14:00:00      high             NaN
206        trace        checkoutservice-0     trace.to_productcatalogservice-0.error_rate             30        2022-03-21 14:00:00      high             NaN
1074       trace  recommendationservice-0     trace.to_productcatalogservice-2.error_rate             30        2022-03-21 14:00:00      high             NaN
1086       trace  recommendationservice-1                trace.from_frontend-2.error_rate             30        2022-03-21 14:00:00      high             NaN
1082       trace  recommendationservice-1                trace.from_frontend-1.error_rate             30        2022-03-21 14:00:00      high             NaN
1070       trace  recommendationservice-0     trace.to_productcatalogservice-1.error_rate             30        2022-03-21 14:00:00      high             NaN
1066       trace  recommendationservice-0     trace.to_productcatalogservice-0.error_rate             30        2022-03-21 14:00:00      high             NaN
253        trace        checkoutservice-1             trace.to_cartservice-1.duration_p95             30        2022-03-21 14:00:00      high             NaN
252        trace        checkoutservice-1            trace.to_cartservice-1.duration_mean             30        2022-03-21 14:00:00      high             NaN
250        trace        checkoutservice-1               trace.to_cartservice-0.error_rate             30        2022-03-21 14:00:00      high             NaN
249        trace        checkoutservice-1             trace.to_cartservice-0.duration_p95             30        2022-03-21 14:00:00      high             NaN
262        trace        checkoutservice-1           trace.to_currencyservice-0.error_rate             30        2022-03-21 14:00:00      high             NaN
926        trace  productcatalogservice-0         trace.from_checkoutservice-1.error_rate             30        2022-03-21 14:00:00      high             NaN
922        trace  productcatalogservice-0         trace.from_checkoutservice-0.error_rate             30        2022-03-21 14:00:00      high             NaN
934        trace  productcatalogservice-0                trace.from_frontend-0.error_rate             30        2022-03-21 14:00:00      high             NaN
918        trace        paymentservice2-0        trace.from_checkoutservice2-0.error_rate             30        2022-03-21 14:00:00      high             NaN
1078       trace  recommendationservice-1                trace.from_frontend-0.error_rate             30        2022-03-21 14:00:00      high             NaN
842        trace               frontend-2           trace.to_shippingservice-2.error_rate             30        2022-03-21 14:00:00      high             NaN
266        trace        checkoutservice-1           trace.to_currencyservice-1.error_rate             30        2022-03-21 14:00:00      high             NaN
958        trace  productcatalogservice-0  trace.from_recommendationservice2-0.error_rate             30        2022-03-21 14:00:00      high             NaN,    source_file                  cmdb_id         kpi_name  anomaly_count earliest_anomaly_timestamp direction  severity_ratio
2          log              adservice-1  log.error_count             31        2022-03-21 14:00:00      high             NaN
6          log            cartservice-0  log.error_count             31        2022-03-21 14:00:00      high             NaN
8          log            cartservice-1  log.error_count             31        2022-03-21 14:00:00      high             NaN
10         log            cartservice-2  log.error_count             31        2022-03-21 14:00:00      high             NaN
18         log        currencyservice-0  log.error_count             31        2022-03-21 14:00:00      high             NaN
20         log        currencyservice-1  log.error_count             31        2022-03-21 14:00:00      high             NaN
22         log        currencyservice-2  log.error_count             31        2022-03-21 14:00:00      high             NaN
50         log  recommendationservice-1  log.error_count             31        2022-03-21 14:00:00      high             NaN
52         log  recommendationservice-2  log.error_count             31        2022-03-21 14:00:00      high             NaN
58         log        shippingservice-1  log.error_count             31        2022-03-21 14:00:00      high             NaN
48         log  recommendationservice-0  log.error_count             31        2022-03-21 14:00:00      high             NaN
30         log               frontend-0  log.error_count             31        2022-03-21 14:00:00      high             NaN
34         log               frontend-2  log.error_count             31        2022-03-21 14:00:00      high             NaN
12         log        checkoutservice-0  log.error_count             30        2022-03-21 14:00:00      high             NaN
16         log        checkoutservice-2  log.error_count             30        2022-03-21 14:00:00      high             NaN
38         log         paymentservice-1  log.error_count             30        2022-03-21 14:00:00      high             NaN
44         log  productcatalogservice-1  log.error_count             30        2022-03-21 14:00:00      high             NaN
40         log         paymentservice-2  log.error_count             30        2022-03-21 14:00:00      high             NaN
56         log        shippingservice-0  log.error_count             30        2022-03-21 14:00:00      high             NaN
60         log        shippingservice-2  log.error_count             30        2022-03-21 14:00:00      high             NaN)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing aggregated anomaly summaries if present, otherwise create empty DataFrames with expected columns
expected_cols = ['source_file','cmdb_id','kpi_name','anomaly_count',
                 'earliest_anomaly_timestamp','latest_anomaly_timestamp',
                 'max_value_in_window','min_value_in_window','direction','severity_ratio']

metric_df = globals().get('metric_anoms_window', pd.DataFrame(columns=expected_cols))
trace_df  = globals().get('trace_anoms_window',  pd.DataFrame(columns=expected_cols))
log_df    = globals().get('log_anoms_window',    pd.DataFrame(columns=expected_cols))

# Ensure source_file column exists and is filled
if 'source_file' not in metric_df.columns:
    metric_df = metric_df.assign(source_file='metric')
if 'source_file' not in trace_df.columns:
    trace_df = trace_df.assign(source_file='trace')
if 'source_file' not in log_df.columns:
    log_df = log_df.assign(source_file='log')

# Standardize column set (add missing expected cols to avoid merge issues)
def ensure_cols(df):
    for c in expected_cols:
        if c not in df.columns:
            df[c] = pd.NA
    return df[expected_cols]
metric_df = ensure_cols(metric_df)
trace_df  = ensure_cols(trace_df)
log_df    = ensure_cols(log_df)

# 1) Concatenate
all_anoms_window = pd.concat([metric_df, trace_df, log_df], ignore_index=True)

# 2) Filter by allowed cmdb_id list (include both pod/service/node names exactly as provided)
allowed = ["node-1","node-2","node-3","node-4","node-5","node-6",
"frontend-0","frontend-1","frontend-2","frontend2-0",
"shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
"checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
"currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
"adservice-0","adservice-1","adservice-2","adservice2-0",
"emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
"cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
"productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
"recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
"paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0",
"frontend","shippingservice","checkoutservice","currencyservice","adservice","emailservice","cartservice",
"productcatalogservice","recommendationservice","paymentservice"]

all_anoms_window = all_anoms_window[all_anoms_window['cmdb_id'].isin(allowed)].copy()

# If there are no rows after filtering, prepare empty summary and return
if all_anoms_window.shape[0] == 0:
    suspicious_components_summary = pd.DataFrame(columns=[
        'cmdb_id','sources_present','number_of_kpis_flagged','total_anomaly_count',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp'
    ])
    all_anoms_window, suspicious_components_summary
else:
    # 3) For each cmdb_id compute aggregates
    # Normalize timestamp columns: parse earliest/latest strings into timezone-aware datetimes for accurate min/max
    def parse_ts_col(s):
        # handle missing/NA
        return pd.to_datetime(s, errors='coerce').dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT') \
               .dt.tz_convert(tz)
    # But since strings likely have no tz, we'll parse as naive then localize
    # Create temporary columns with parsed datetimes for aggregation
    all_anoms_window['_earliest_dt'] = pd.to_datetime(all_anoms_window['earliest_anomaly_timestamp'], errors='coerce')
    all_anoms_window['_latest_dt']   = pd.to_datetime(all_anoms_window['latest_anomaly_timestamp'], errors='coerce')
    # Localize naive datetimes to Asia/Shanghai
    def localize_series(s):
        # s may contain NaT
        return s.apply(lambda x: tz.localize(x) if pd.notna(x) and x.tzinfo is None else x)
    all_anoms_window['_earliest_dt'] = localize_series(all_anoms_window['_earliest_dt'])
    all_anoms_window['_latest_dt']   = localize_series(all_anoms_window['_latest_dt'])

    # Compute sources_present (as sorted list string), number_of_kpis_flagged (distinct kpi_name), total_anomaly_count (sum), earliest/latest timestamps across KPIs
    grouped = all_anoms_window.groupby('cmdb_id').agg(
        sources_present_list = ('source_file', lambda x: sorted(x.dropna().unique().tolist())),
        number_of_kpis_flagged = ('kpi_name', lambda x: x.dropna().nunique()),
        total_anomaly_count = ('anomaly_count', 'sum'),
        earliest_dt = ('_earliest_dt', 'min'),
        latest_dt = ('_latest_dt', 'max')
    ).reset_index()

    # Convert sources_present_list to comma-joined string for compactness and also store count
    grouped['sources_present'] = grouped['sources_present_list'].apply(lambda lst: ','.join(lst) if isinstance(lst, list) else '')
    grouped['sources_present_count'] = grouped['sources_present_list'].apply(lambda lst: len(lst) if isinstance(lst, list) else 0)

    # Format earliest/latest timestamps back to string in Asia/Shanghai
    def fmt(dt):
        if pd.isna(dt):
            return pd.NA
        # ensure tz-aware
        if dt.tzinfo is None:
            dt = tz.localize(dt)
        return dt.astimezone(tz).strftime('%Y-%m-%d %H:%M:%S')

    grouped['earliest_anomaly_timestamp'] = grouped['earliest_dt'].apply(fmt)
    grouped['latest_anomaly_timestamp']   = grouped['latest_dt'].apply(fmt)

    # Select desired columns and sort by total_anomaly_count desc
    suspicious_components_summary = grouped[[
        'cmdb_id','sources_present','number_of_kpis_flagged','total_anomaly_count',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp'
    ]].sort_values('total_anomaly_count', ascending=False).reset_index(drop=True)

    # 5) For top 5 cmdb_id, return their top 5 anomalous kpis
    top_components = suspicious_components_summary.head(5)['cmdb_id'].tolist()

    # From all_anoms_window, pick rows for these components and sort by anomaly_count desc, then take top 5 per component
    top_kpis_list = []
    for comp in top_components:
        comp_df = all_anoms_window[all_anoms_window['cmdb_id'] == comp].copy()
        if comp_df.shape[0] == 0:
            continue
        # Ensure anomaly_count numeric
        comp_df['anomaly_count'] = pd.to_numeric(comp_df['anomaly_count'], errors='coerce').fillna(0).astype(int)
        comp_top = comp_df.sort_values('anomaly_count', ascending=False).head(5)
        # Select required columns: include earliest_anomaly_timestamp as string (original)
        comp_top_small = comp_top[[
            'cmdb_id','kpi_name','source_file','anomaly_count','direction','severity_ratio','earliest_anomaly_timestamp'
        ]].copy()
        top_kpis_list.append(comp_top_small)

    if len(top_kpis_list) > 0:
        top_kpis_for_top5 = pd.concat(top_kpis_list, ignore_index=True)
    else:
        top_kpis_for_top5 = pd.DataFrame(columns=['cmdb_id','kpi_name','source_file','anomaly_count','direction','severity_ratio','earliest_anomaly_timestamp'])

    # Store concatenated anomalies into kernel variable (already in var all_anoms_window) and component summary
    globals()['all_anoms_window'] = all_anoms_window
    globals()['suspicious_components_summary'] = suspicious_components_summary

    # Return compact bounded results: top 20 component summary rows and top_kpis_for_top5 (max 25 rows)
    suspicious_components_summary.head(20), top_kpis_for_top5

```
Out[4]:
```
```python
# Summarize the concatenated anomaly analysis results in plain English.
# This is a summarization task — produce a concise English summary string using existing kernel variables.

import pandas as pd

suspicious = globals().get('suspicious_components_summary', None)
all_anoms = globals().get('all_anoms_window', None)

if suspicious is None or suspicious.shape[0] == 0:
    summary = "No suspicious components found among the allowed list for the analyzed window."
else:
    total_components = suspicious.shape[0]
    top20 = suspicious.head(20)
    top5_ids = top20.head(5)['cmdb_id'].tolist()

    lines = []
    lines.append(f"Found {total_components} suspicious component(s) within the allowed list.")
    lines.append("Top components by total anomaly count (up to 20):")
    for _, r in top20.iterrows():
        lines.append(
            f"- {r['cmdb_id']}: total_anomaly_count={int(r['total_anomaly_count'])}, "
            f"kpis_flagged={int(r['number_of_kpis_flagged'])}, sources_present={r['sources_present']}, "
            f"earliest={r['earliest_anomaly_timestamp']}, latest={r['latest_anomaly_timestamp']}"
        )

    # Top 5 components' top 5 anomalous KPIs
    lines.append("\nFor the top 5 components, top anomalous KPIs (kpi_name | source_file | anomaly_count | direction | severity_ratio | earliest):")
    for comp in top5_ids:
        comp_rows = all_anoms[all_anoms['cmdb_id'] == comp].copy() if all_anoms is not None else pd.DataFrame()
        if comp_rows.shape[0] == 0:
            lines.append(f"- {comp}: (no anomaly rows available)")
            continue
        comp_rows['anomaly_count'] = pd.to_numeric(comp_rows['anomaly_count'], errors='coerce').fillna(0).astype(int)
        comp_topk = comp_rows.sort_values('anomaly_count', ascending=False).head(5)
        lines.append(f"- {comp}:")
        for _, k in comp_topk.iterrows():
            sev = k['severity_ratio']
            sev_str = f"{sev:.3g}" if pd.notna(sev) else "NaN"
            kpi_short = (k['kpi_name'][:80] + '...') if len(str(k['kpi_name']))>83 else k['kpi_name']
            lines.append(
                f"    * {kpi_short} | {k['source_file']} | count={int(k['anomaly_count'])} | dir={k['direction']} | sev={sev_str} | earliest={k['earliest_anomaly_timestamp']}"
            )

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing aggregated anomaly summaries from kernel or create empty DataFrames with expected columns
expected_cols = ['source_file','cmdb_id','kpi_name','anomaly_count',
                 'earliest_anomaly_timestamp','latest_anomaly_timestamp',
                 'max_value_in_window','min_value_in_window','direction','severity_ratio']

metric_df = globals().get('metric_anoms_window', pd.DataFrame(columns=expected_cols))
trace_df  = globals().get('trace_anoms_window',  pd.DataFrame(columns=expected_cols))
log_df    = globals().get('log_anoms_window',    pd.DataFrame(columns=expected_cols))

# Ensure source_file column exists and is correctly set
if 'source_file' not in metric_df.columns:
    metric_df = metric_df.assign(source_file='metric')
else:
    metric_df['source_file'] = metric_df['source_file'].fillna('metric')

if 'source_file' not in trace_df.columns:
    trace_df = trace_df.assign(source_file='trace')
else:
    trace_df['source_file'] = trace_df['source_file'].fillna('trace')

if 'source_file' not in log_df.columns:
    log_df = log_df.assign(source_file='log')
else:
    log_df['source_file'] = log_df['source_file'].fillna('log')

# 1) Concatenate into all_anoms_window
all_anoms_window = pd.concat([metric_df, trace_df, log_df], ignore_index=True, sort=False)

# 2) Filter by exact allowed cmdb_id list
allowed = ["node-1","node-2","node-3","node-4","node-5","node-6",
"frontend-0","frontend-1","frontend-2","frontend2-0",
"shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
"checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
"currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
"adservice-0","adservice-1","adservice-2","adservice2-0",
"emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
"cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
"productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
"recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
"paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0",
"frontend","shippingservice","checkoutservice","currencyservice","adservice","emailservice","cartservice",
"productcatalogservice","recommendationservice","paymentservice"]

all_anoms_window = all_anoms_window[all_anoms_window['cmdb_id'].isin(allowed)].copy()

# 3) Aggregate by cmdb_id
# Ensure anomaly_count numeric
all_anoms_window['anomaly_count'] = pd.to_numeric(all_anoms_window.get('anomaly_count', 0), errors='coerce').fillna(0).astype(int)

# Parse earliest/latest timestamp strings to datetime, localize to Asia/Shanghai if naive
def parse_local_ts(s):
    dt = pd.to_datetime(s, errors='coerce')
    if pd.isna(dt).any():
        # return Series to handle NaT gracefully
        return dt
    # If tz-naive, localize; if tz-aware, convert to Asia/Shanghai
    def _localize(x):
        if pd.isna(x):
            return pd.NaT
        if x.tzinfo is None:
            return tz.localize(x)
        else:
            return x.astimezone(tz)
    return dt.map(_localize)

all_anoms_window['_earliest_dt'] = parse_local_ts(all_anoms_window.get('earliest_anomaly_timestamp'))
all_anoms_window['_latest_dt']   = parse_local_ts(all_anoms_window.get('latest_anomaly_timestamp'))

grouped = all_anoms_window.groupby('cmdb_id').agg(
    sources_present_list = ('source_file', lambda x: sorted(x.dropna().unique().tolist())),
    number_of_kpis_flagged = ('kpi_name', lambda x: x.dropna().nunique()),
    total_anomaly_count = ('anomaly_count', 'sum'),
    earliest_dt = ('_earliest_dt', 'min'),
    latest_dt = ('_latest_dt', 'max')
).reset_index()

# Format sources_present as comma-joined string
grouped['sources_present'] = grouped['sources_present_list'].apply(lambda lst: ','.join(lst) if isinstance(lst, list) else '')

# Format earliest/latest to string in Asia/Shanghai
def fmt_dt(x):
    if pd.isna(x):
        return pd.NA
    if x.tzinfo is None:
        x = tz.localize(x)
    return x.astimezone(tz).strftime('%Y-%m-%d %H:%M:%S')

grouped['earliest_anomaly_timestamp'] = grouped['earliest_dt'].apply(fmt_dt)
grouped['latest_anomaly_timestamp']   = grouped['latest_dt'].apply(fmt_dt)

suspicious_components_summary = grouped[[
    'cmdb_id','sources_present','number_of_kpis_flagged','total_anomaly_count',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp'
]].sort_values('total_anomaly_count', ascending=False).reset_index(drop=True)

# 4) Store the filtered concatenated rows and the aggregated summary into the kernel
globals()['all_anoms_window'] = all_anoms_window
globals()['suspicious_components_summary'] = suspicious_components_summary

# 5) Prepare compact bounded outputs
suspicious_components_summary_top20 = suspicious_components_summary.head(20)

# For top 5 cmdb_id, get up to top 5 anomalous KPIs each
top5_components = suspicious_components_summary_top20.head(5)['cmdb_id'].tolist()

top_kpis_list = []
for comp in top5_components:
    comp_rows = all_anoms_window[all_anoms_window['cmdb_id'] == comp].copy()
    if comp_rows.shape[0] == 0:
        continue
    comp_rows['anomaly_count'] = pd.to_numeric(comp_rows['anomaly_count'], errors='coerce').fillna(0).astype(int)
    comp_topk = comp_rows.sort_values('anomaly_count', ascending=False).head(5)
    top_kpis_list.append(comp_topk[[
        'cmdb_id','kpi_name','source_file','anomaly_count','direction','severity_ratio','earliest_anomaly_timestamp'
    ]])

if top_kpis_list:
    top_kpis_for_top5 = pd.concat(top_kpis_list, ignore_index=True)
else:
    top_kpis_for_top5 = pd.DataFrame(columns=['cmdb_id','kpi_name','source_file','anomaly_count','direction','severity_ratio','earliest_anomaly_timestamp'])

# Return compact results (two variables)
suspicious_components_summary_top20, top_kpis_for_top5

```
Out[5]:
```
```python
# Summarize the concatenation and aggregation results in plain English using existing kernel variables.
import pandas as pd

suspicious = globals().get('suspicious_components_summary')
top_kpis = globals().get('top_kpis_for_top5')

if suspicious is None or suspicious.shape[0] == 0:
    summary = "No suspicious components found in the allowed list for the analyzed window."
else:
    total_components = int(suspicious.shape[0])
    # Top 10 components for concise summary
    top_components = suspicious.head(10)
    lines = []
    lines.append(f"Found {total_components} suspicious component(s) within the allowed root-cause candidates.")
    lines.append("Top components by total anomaly count (showing up to 10):")
    for _, r in top_components.iterrows():
        lines.append(
            f"- {r['cmdb_id']}: total_anomaly_count={int(r['total_anomaly_count'])}, "
            f"kpis_flagged={int(r['number_of_kpis_flagged'])}, sources_present={r['sources_present']}, "
            f"earliest={r['earliest_anomaly_timestamp']}, latest={r['latest_anomaly_timestamp']}"
        )
    # Top KPIs for top 5 components
    if top_kpis is not None and top_kpis.shape[0] > 0:
        lines.append("\nTop anomalous KPIs for the top 5 components (up to 5 KPIs per component):")
        # group by component to display up to 5 each
        for comp in top_components.head(5)['cmdb_id'].tolist():
            comp_rows = top_kpis[top_kpis['cmdb_id'] == comp]
            if comp_rows.shape[0] == 0:
                lines.append(f"- {comp}: (no KPI rows available)")
                continue
            lines.append(f"- {comp}:")
            for _, k in comp_rows.iterrows():
                sev = k['severity_ratio']
                sev_str = f"{sev:.3g}" if pd.notna(sev) else "NaN"
                kpi_name = str(k['kpi_name'])
                if len(kpi_name) > 80:
                    kpi_name = kpi_name[:77] + '...'
                lines.append(
                    f"    * {kpi_name} | {k['source_file']} | count={int(k['anomaly_count'])} | dir={k['direction']} | sev={sev_str} | earliest={k['earliest_anomaly_timestamp']}"
                )
    else:
        lines.append("\nNo detailed top-KPI rows available for top components.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id   sources_present  number_of_kpis_flagged  total_anomaly_count earliest_anomaly_timestamp latest_anomaly_timestamp
0                  adservice            metric                     341                 9542        2022-03-21 14:00:00      2022-03-21 14:30:00
1                 frontend-0  log,metric,trace                     286                 5527        2022-03-21 14:00:00      2022-03-21 14:30:00
2                 frontend-2  log,metric,trace                     283                 5303        2022-03-21 14:00:00      2022-03-21 14:30:00
3                 frontend-1  log,metric,trace                     284                 5276        2022-03-21 14:00:00      2022-03-21 14:30:00
4                frontend2-0      metric,trace                     240                 4932        2022-03-21 14:00:00      2022-03-21 14:30:00
5          checkoutservice-1  log,metric,trace                     242                 4838        2022-03-21 14:00:00      2022-03-21 14:30:00
6          checkoutservice-0  log,metric,trace                     238                 4524        2022-03-21 14:00:00      2022-03-21 14:30:00
7          checkoutservice-2  log,metric,trace                     238                 4332        2022-03-21 14:00:00      2022-03-21 14:30:00
8         checkoutservice2-0      metric,trace                     184                 3622        2022-03-21 14:00:00      2022-03-21 14:30:00
9    productcatalogservice-1  log,metric,trace                     162                 3155        2022-03-21 14:00:00      2022-03-21 14:30:00
10   productcatalogservice-0  log,metric,trace                     162                 3053        2022-03-21 14:00:00      2022-03-21 14:30:00
11   productcatalogservice-2  log,metric,trace                     158                 3007        2022-03-21 14:00:00      2022-03-21 14:30:00
12             cartservice-2  log,metric,trace                     130                 2922        2022-03-21 14:00:00      2022-03-21 14:30:00
13             cartservice-0  log,metric,trace                     130                 2896        2022-03-21 14:00:00      2022-03-21 14:30:00
14   recommendationservice-2  log,metric,trace                     128                 2778        2022-03-21 14:00:00      2022-03-21 14:30:00
15   recommendationservice-1  log,metric,trace                     128                 2765        2022-03-21 14:00:00      2022-03-21 14:30:00
16   recommendationservice-0  log,metric,trace                     128                 2743        2022-03-21 14:00:00      2022-03-21 14:30:00
17  recommendationservice2-0      metric,trace                     122                 2737        2022-03-21 14:00:00      2022-03-21 14:30:00
18             cartservice-1  log,metric,trace                     122                 2708        2022-03-21 14:00:00      2022-03-21 14:30:00
19              adservice2-0      metric,trace                     106                 2552        2022-03-21 14:00:00      2022-03-21 14:30:00,         cmdb_id                                           kpi_name source_file  anomaly_count direction  severity_ratio earliest_anomaly_timestamp
0     adservice                  runtime.jvm_threads_state.WAITING      metric             31      high        1.000000        2022-03-21 14:00:00
1     adservice               runtime.jvm_threads_state.TERMINATED      metric             31      high             NaN        2022-03-21 14:00:00
2     adservice                 runtime.jvm_threads_state.RUNNABLE      metric             31      high        1.016667        2022-03-21 14:00:00
3     adservice                                        app.grpc.rr      metric             31      high        1.000000        2022-03-21 14:00:00
4     adservice                                        app.grpc.sr      metric             31      high        1.000000        2022-03-21 14:00:00
..          ...                                                ...         ...            ...       ...             ...                        ...
20  frontend2-0    container.node-6.container_cpu_load_average_10s      metric             31      high             NaN        2022-03-21 14:00:00
21  frontend2-0        container.node-6.container_file_descriptors      metric             31      high        1.000000        2022-03-21 14:00:00
22  frontend2-0   container.node-6.container_fs_limit_MB./dev/vda1      metric             31      high        1.000000        2022-03-21 14:00:00
23  frontend2-0  container.node-6.container_fs_io_time_weighted...      metric             31      high             NaN        2022-03-21 14:00:00
24  frontend2-0  container.node-6.container_fs_io_time_seconds....      metric             31      high             NaN        2022-03-21 14:00:00

[25 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse metric_window_rows from kernel
metric_rows = globals().get('metric_window_rows')
if metric_rows is None:
    raise RuntimeError("metric_window_rows not found in kernel")

tz = pytz.timezone('Asia/Shanghai')  # UTC+8

# 1) Focused components list (exact match)
focus_components = [
    'adservice','frontend-0','frontend-1','frontend-2','frontend2-0',
    'checkoutservice-0','checkoutservice-1','checkoutservice-2'
]

# Filter rows for those components
df_focus = metric_rows[metric_rows['cmdb_id'].isin(focus_components)].copy()

# Ensure anomaly boolean column exists and timestamps are int
if 'anomaly' not in df_focus.columns:
    raise RuntimeError("metric_window_rows does not contain 'anomaly' column")
df_focus['timestamp'] = df_focus['timestamp'].astype(int)
df_focus['anomaly'] = df_focus['anomaly'].astype(bool)

# 2) Detect consecutive-anomaly segments per (cmdb_id, kpi_name)
segments = []
group_cols = ['cmdb_id', 'kpi_name']
for (cmdb, kpi), grp in df_focus.groupby(group_cols):
    # sort by timestamp
    grp_sorted = grp.sort_values('timestamp', ascending=True)
    # take only anomaly==True rows
    anom = grp_sorted[grp_sorted['anomaly']].copy()
    if anom.empty:
        continue
    # compute diff between consecutive anomaly timestamps
    anom['diff'] = anom['timestamp'].diff().fillna(-1).astype(int)
    # start new segment where diff != 60
    anom['segment_id'] = (anom['diff'] != 60).cumsum()
    # aggregate segments
    agg = anom.groupby('segment_id').agg(
        cmdb_id = ('cmdb_id', 'first'),
        kpi_name = ('kpi_name', 'first'),
        segment_start_ts = ('timestamp', 'min'),
        segment_end_ts = ('timestamp', 'max'),
        segment_length_minutes = ('timestamp', 'count'),
        max_value_in_segment = ('value', 'max'),
        p95_threshold = ('p95', 'first')
    ).reset_index(drop=True)
    if agg.empty:
        continue
    # compute severity_ratio and direction
    def compute_sev_dir(row):
        p95 = row['p95_threshold']
        maxv = row['max_value_in_segment']
        if pd.isna(p95) or p95 == 0:
            sev = float('nan')
        else:
            sev = maxv / p95
        direction = 'high' if (not pd.isna(p95) and maxv >= p95) else 'low'
        return pd.Series({'severity_ratio': sev, 'direction': direction})
    sd = agg.apply(compute_sev_dir, axis=1)
    agg = pd.concat([agg, sd], axis=1)
    # filter out short segments (<3 minutes)
    agg = agg[agg['segment_length_minutes'] >= 3]
    if not agg.empty:
        segments.append(agg)

# 3) Concatenate all segments and store into kernel variable metric_faults_top
if segments:
    metric_faults_top = pd.concat(segments, ignore_index=True)
else:
    # empty with expected columns
    metric_faults_top = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','segment_start_ts','segment_end_ts',
        'segment_length_minutes','max_value_in_segment','p95_threshold','severity_ratio','direction'
    ])

# Convert timestamps to Asia/Shanghai formatted strings for start/end
def fmt_ts(ts):
    try:
        return datetime.fromtimestamp(int(ts), tz).strftime('%Y-%m-%d %H:%M:%S')
    except Exception:
        return pd.NA

metric_faults_top['segment_start_timestamp'] = metric_faults_top['segment_start_ts'].apply(fmt_ts)
metric_faults_top['segment_end_timestamp'] = metric_faults_top['segment_end_ts'].apply(fmt_ts)

# Select and order final columns
metric_faults_top = metric_faults_top[[
    'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
    'segment_length_minutes','max_value_in_segment','p95_threshold','direction','severity_ratio'
]]

# 4) Store into kernel (already assigned)
globals()['metric_faults_top'] = metric_faults_top

# 5) Prepare compact display: truncate kpi_name to 100 chars, sort by length desc then severity desc, limit 50 rows
def trunc_kpi(s, n=100):
    s = str(s)
    return s if len(s) <= n else s[:n-3] + '...'

display_df = metric_faults_top.copy()
display_df['kpi_name'] = display_df['kpi_name'].apply(lambda x: trunc_kpi(x, 100))
# severity_ratio may be NaN; sort with NaNs last
display_df = display_df.sort_values(['segment_length_minutes','severity_ratio'], ascending=[False, False], na_position='last')
display_top50 = display_df.head(50)[[
    'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
    'segment_length_minutes','direction','severity_ratio'
]]

# Return compact result variable (IPython will display)
display_top50

```
Out[6]:
```
```python
# Summarize fault segments detected in metric_faults_top for the focused components.
import pandas as pd
metric_faults_top = globals().get('metric_faults_top', pd.DataFrame())

if metric_faults_top is None or metric_faults_top.shape[0] == 0:
    summary = "No fault segments (>=3 minutes) found for the focused components in metric_window_rows."
else:
    # total segments
    total_segments = int(metric_faults_top.shape[0])
    # overall earliest start and latest end
    earliest_start = metric_faults_top['segment_start_timestamp'].min()
    latest_end = metric_faults_top['segment_end_timestamp'].max()
    # group by component
    grp = metric_faults_top.groupby('cmdb_id').agg(
        segments_count = ('segment_length_minutes', 'size'),
        max_segment_length = ('segment_length_minutes', 'max')
    ).reset_index().sort_values(['segments_count','max_segment_length'], ascending=False)
    top3 = grp.head(3)
    top3_lines = []
    for _, r in top3.iterrows():
        top3_lines.append(f"{r['cmdb_id']} (segments={int(r['segments_count'])}, longest={int(r['max_segment_length'])} min)")
    # find maximum severity_ratio row (non-NaN)
    metric_faults_top['severity_ratio_num'] = pd.to_numeric(metric_faults_top['severity_ratio'], errors='coerce')
    max_sev_row = metric_faults_top.loc[metric_faults_top['severity_ratio_num'].idxmax()] if metric_faults_top['severity_ratio_num'].notna().any() else None
    max_sev_info = ""
    if isinstance(max_sev_row, pd.Series):
        max_sev_info = (f"{max_sev_row['cmdb_id']} | {str(max_sev_row['kpi_name'])[:80]}... | "
                        f"severity_ratio={max_sev_row['severity_ratio_num']:.3g} | "
                        f"length={int(max_sev_row['segment_length_minutes'])} min")
    else:
        max_sev_info = "No numeric severity_ratio available (all NaN or p95==0)."

    # count how many segments span the entire 31-minute window
    full_window_count = int((metric_faults_top['segment_length_minutes'] >= 31).sum())

    summary = (
        f"Fault-segment summary for selected components:\n"
        f"- Total fault segments (>=3 min): {total_segments}\n"
        f"- Time window covered by segments: earliest_start={earliest_start}, latest_end={latest_end}\n"
        f"- Segments spanning the full 31-minute window: {full_window_count}\n"
        f"- Top components by segments/longest segment (top 3): " + "; ".join(top3_lines) + "\n"
        f"- Highest observed severity_ratio (numeric): {max_sev_info}\n"
        f"- Notes: Most detected segments are continuous anomalies (one-minute spacing) and many last the full 31 minutes.\n"
        f"  severity_ratio = max_value_in_segment / p95_threshold where p95>0; where p95==0 or missing, severity is NaN.\n"
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name segment_start_timestamp segment_end_timestamp  segment_length_minutes direction  severity_ratio
664  checkoutservice-2                 container.node-5.container_sockets     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.250000
627  checkoutservice-2        container.node-5.container_file_descriptors     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.100000
334          adservice                 runtime.jvm_threads_state.RUNNABLE     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.016667
750         frontend-0         container.node-6.container_memory_usage_MB     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.000903
1            adservice                                        app.grpc.rr     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.000000
..                 ...                                                ...                     ...                   ...                     ...       ...             ...
51           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.000000
52           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.000000
53           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.000000
54           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.000000
55           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     2022-03-21 14:00:00   2022-03-21 14:30:00                      31      high        1.000000

[50 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables
metric_faults = globals().get('metric_faults_top', pd.DataFrame())
trace_anoms = globals().get('trace_anoms_window', pd.DataFrame())

# 1) Select metric_faults_top rows where kpi_name contains 'node-5' or 'node-6'
# Ensure kpi_name column exists
if 'kpi_name' not in metric_faults.columns:
    metric_faults = pd.DataFrame(columns=['cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp','segment_length_minutes','severity_ratio'])

# Filter by substring match
mask_node5 = metric_faults['kpi_name'].str.contains('node-5', na=False)
mask_node6 = metric_faults['kpi_name'].str.contains('node-6', na=False)
node_rows = metric_faults[mask_node5 | mask_node6].copy()

# Map to node_id
def pick_node_id(k):
    k = str(k)
    if 'node-5' in k:
        return 'node-5'
    if 'node-6' in k:
        return 'node-6'
    return pd.NA

node_rows['node_id'] = node_rows['kpi_name'].apply(pick_node_id)

# Parse start/end timestamps to datetimes localized to Asia/Shanghai for accurate min/max
def parse_and_localize(col):
    dt = pd.to_datetime(node_rows[col], errors='coerce')
    # localize naive datetimes
    def loc(x):
        if pd.isna(x):
            return pd.NaT
        if x.tzinfo is None:
            return tz.localize(x)
        return x.astimezone(tz)
    return dt.map(loc)

if 'segment_start_timestamp' in node_rows.columns and 'segment_end_timestamp' in node_rows.columns:
    node_rows['_start_dt'] = parse_and_localize('segment_start_timestamp')
    node_rows['_end_dt'] = parse_and_localize('segment_end_timestamp')
else:
    node_rows['_start_dt'] = pd.NaT
    node_rows['_end_dt'] = pd.NaT

# 1) Aggregate per node (node-5 and node-6)
agg_node = node_rows.groupby('node_id').agg(
    number_of_fault_segments = ('kpi_name', 'size'),
    number_of_distinct_cmdb_ids_affected = ('cmdb_id', lambda x: x.dropna().nunique()),
    earliest_segment_start = ('_start_dt', 'min'),
    latest_segment_end = ('_end_dt', 'max'),
    count_of_kpi_names = ('kpi_name', lambda x: x.dropna().nunique())
).reset_index()

# Format earliest/latest as strings in Asia/Shanghai
def fmt_dt(x):
    if pd.isna(x):
        return pd.NA
    return x.astimezone(tz).strftime('%Y-%m-%d %H:%M:%S')

agg_node['earliest_segment_start'] = agg_node['earliest_segment_start'].apply(fmt_dt)
agg_node['latest_segment_end'] = agg_node['latest_segment_end'].apply(fmt_dt)

# Keep only required columns and ensure order
node5_node6_metric_summary = agg_node[[
    'node_id','number_of_fault_segments','number_of_distinct_cmdb_ids_affected',
    'earliest_segment_start','latest_segment_end','count_of_kpi_names'
]].rename(columns={'node_id':'node_id','count_of_kpi_names':'count_of_kpi_names'})

# 1) Example KPI rows: up to 10 per node, sorted by segment_length_minutes desc then severity_ratio desc
# Ensure numeric sorting columns
node_rows['segment_length_minutes'] = pd.to_numeric(node_rows.get('segment_length_minutes', 0), errors='coerce').fillna(0).astype(int)
node_rows['severity_ratio_num'] = pd.to_numeric(node_rows.get('severity_ratio', pd.NA), errors='coerce')

examples_list = []
for node in ['node-5','node-6']:
    comp = node_rows[node_rows['node_id'] == node].copy()
    if comp.empty:
        continue
    comp_sorted = comp.sort_values(['segment_length_minutes','severity_ratio_num'], ascending=[False, False], na_position='last')
    ex = comp_sorted.head(10)[[
        'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp','segment_length_minutes','severity_ratio'
    ]].copy()
    ex['node_id'] = node
    examples_list.append(ex)

if examples_list:
    node5_node6_example_kpis = pd.concat(examples_list, ignore_index=True)
    # Cap total examples at 20 (as requested)
    node5_node6_example_kpis = node5_node6_example_kpis.head(20)
else:
    node5_node6_example_kpis = pd.DataFrame(columns=['cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp','segment_length_minutes','severity_ratio','node_id'])

# 2) From trace_anoms_window select rows with 'error_rate' in kpi_name and earliest_anomaly_timestamp in window
trace = trace_anoms.copy() if trace_anoms is not None else pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_timestamp'])
if 'kpi_name' in trace.columns and 'earliest_anomaly_timestamp' in trace.columns:
    trace_mask = trace['kpi_name'].str.contains('error_rate', na=False)
    trace_err = trace[trace_mask].copy()
    # parse earliest_anomaly_timestamp and localize to Asia/Shanghai
    trace_err['_earliest_dt'] = pd.to_datetime(trace_err['earliest_anomaly_timestamp'], errors='coerce')
    # localize naive timestamps
    def loc_trace(x):
        if pd.isna(x):
            return pd.NaT
        if x.tzinfo is None:
            return tz.localize(x)
        return x.astimezone(tz)
    trace_err['_earliest_dt'] = trace_err['_earliest_dt'].map(loc_trace)

    # Window bounds
    start = tz.localize(pd.to_datetime('2022-03-21 14:00:00'))
    end   = tz.localize(pd.to_datetime('2022-03-21 14:30:00'))
    # Filter by window (inclusive)
    trace_err_window = trace_err[(trace_err['_earliest_dt'] >= start) & (trace_err['_earliest_dt'] <= end)].copy()

    # Aggregate by cmdb_id
    trace_error_rate_summary = trace_err_window.groupby('cmdb_id').agg(
        count_of_error_rate_kpis_flagged = ('kpi_name', lambda x: x.dropna().nunique()),
        earliest_error_rate_timestamp = ('_earliest_dt', 'min'),
        latest_error_rate_timestamp = ('_earliest_dt', 'max')
    ).reset_index()

    # Format timestamps to strings
    trace_error_rate_summary['earliest_error_rate_timestamp'] = trace_error_rate_summary['earliest_error_rate_timestamp'].apply(lambda x: x.astimezone(tz).strftime('%Y-%m-%d %H:%M:%S') if pd.notna(x) else pd.NA)
    trace_error_rate_summary['latest_error_rate_timestamp'] = trace_error_rate_summary['latest_error_rate_timestamp'].apply(lambda x: x.astimezone(tz).strftime('%Y-%m-%d %H:%M:%S') if pd.notna(x) else pd.NA)

    trace_error_rate_summary = trace_error_rate_summary[[
        'cmdb_id','count_of_error_rate_kpis_flagged','earliest_error_rate_timestamp','latest_error_rate_timestamp'
    ]].sort_values('count_of_error_rate_kpis_flagged', ascending=False).head(20).reset_index(drop=True)
else:
    trace_error_rate_summary = pd.DataFrame(columns=['cmdb_id','count_of_error_rate_kpis_flagged','earliest_error_rate_timestamp','latest_error_rate_timestamp'])

# 4) Store results into kernel variables
globals()['node5_node6_metric_summary'] = node5_node6_metric_summary
globals()['node5_node6_example_kpis'] = node5_node6_example_kpis
globals()['trace_error_rate_summary'] = trace_error_rate_summary

# 5) Return compact bounded tables (as variables in tuple so IPython displays them)
node5_node6_metric_summary, trace_error_rate_summary, node5_node6_example_kpis

```
Out[7]:
```
```python
# Summarize evidence for node-5 and node-6 failures using existing kernel variables.
node_summary = globals().get('node5_node6_metric_summary')
examples = globals().get('node5_node6_example_kpis')
trace_errors = globals().get('trace_error_rate_summary')

def short(s, n=80):
    s = str(s)
    return s if len(s) <= n else s[:n-3] + '...'

if node_summary is None or node_summary.shape[0] == 0:
    summary = "No node-level fault segments found for node-5 or node-6."
else:
    # Extract rows for node-5 and node-6
    def row_for(node):
        r = node_summary[node_summary['node_id'] == node]
        if r.shape[0]==0:
            return None
        r = r.iloc[0]
        return {
            'node': node,
            'segments': int(r['number_of_fault_segments']),
            'distinct_cmdb_ids': int(r['number_of_distinct_cmdb_ids_affected']),
            'earliest': r['earliest_segment_start'],
            'latest': r['latest_segment_end'],
            'distinct_kpis': int(r['count_of_kpi_names'])
        }
    n5 = row_for('node-5')
    n6 = row_for('node-6')

    lines = []
    lines.append("Node-level failure evidence (compact):")
    if n5:
        lines.append(f"- node-5: {n5['segments']} fault segments across {n5['distinct_cmdb_ids']} affected cmdb_id(s); "
                     f"{n5['distinct_kpis']} distinct KPI names flagged; window {n5['earliest']} to {n5['latest']}.")
    else:
        lines.append("- node-5: no segments found.")

    if n6:
        lines.append(f"- node-6: {n6['segments']} fault segments across {n6['distinct_cmdb_ids']} affected cmdb_id(s); "
                     f"{n6['distinct_kpis']} distinct KPI names flagged; window {n6['earliest']} to {n6['latest']}.")
    else:
        lines.append("- node-6: no segments found.")

    # Examples: up to 10 total (already limited) grouped by node
    lines.append("\nExample KPI fault segments (up to 10 total, sorted by length then severity):")
    if examples is None or examples.shape[0]==0:
        lines.append("(no example KPI rows available)")
    else:
        # Show up to 10 rows; include node_id where available
        ex = examples.copy().head(20)  # examples was already capped, but be safe
        # Ensure node_id column present
        if 'node_id' not in ex.columns:
            ex['node_id'] = ex['kpi_name'].apply(lambda k: 'node-5' if 'node-5' in str(k) else ('node-6' if 'node-6' in str(k) else ''))
        for _, r in ex.head(10).iterrows():
            lines.append(f"  * {r.get('node_id','')}: {r['cmdb_id']} | {short(r['kpi_name'],100)} | "
                         f"{r['segment_start_timestamp']} - {r['segment_end_timestamp']} | len={int(r['segment_length_minutes'])} min | sev={r['severity_ratio']}")

    # Trace error_rate evidence
    lines.append("\nTrace-level error_rate anomalies (selected services/pods):")
    if trace_errors is None or trace_errors.shape[0]==0:
        lines.append("(no trace error_rate anomalies in window)")
    else:
        # Show top rows (up to 20) summarizing counts
        top_trace = trace_errors.head(20)
        for _, r in top_trace.iterrows():
            lines.append(f"  * {r['cmdb_id']}: {int(r['count_of_error_rate_kpis_flagged'])} error_rate KPI(s) flagged; "
                         f"earliest={r['earliest_error_rate_timestamp']}, latest={r['latest_error_rate_timestamp']}")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  node_id  number_of_fault_segments  number_of_distinct_cmdb_ids_affected earliest_segment_start   latest_segment_end  count_of_kpi_names
0  node-5                       169                                     3    2022-03-21 14:00:00  2022-03-21 14:30:00                  58
1  node-6                       222                                     4    2022-03-21 14:00:00  2022-03-21 14:30:00                  57,                     cmdb_id  count_of_error_rate_kpis_flagged earliest_error_rate_timestamp latest_error_rate_timestamp
0                frontend-1                                23           2022-03-21 14:00:00         2022-03-21 14:00:00
1         checkoutservice-2                                23           2022-03-21 14:00:00         2022-03-21 14:00:00
2                frontend-0                                23           2022-03-21 14:00:00         2022-03-21 14:00:00
3                frontend-2                                23           2022-03-21 14:00:00         2022-03-21 14:00:00
4         checkoutservice-1                                23           2022-03-21 14:00:00         2022-03-21 14:00:00
5         checkoutservice-0                                23           2022-03-21 14:00:00         2022-03-21 14:00:00
6   productcatalogservice-1                                10           2022-03-21 14:00:00         2022-03-21 14:00:00
7   productcatalogservice-2                                10           2022-03-21 14:00:00         2022-03-21 14:00:00
8   productcatalogservice-0                                10           2022-03-21 14:00:00         2022-03-21 14:00:00
9               frontend2-0                                 9           2022-03-21 14:00:00         2022-03-21 14:00:00
10       checkoutservice2-0                                 9           2022-03-21 14:00:00         2022-03-21 14:00:00
11  recommendationservice-0                                 7           2022-03-21 14:00:00         2022-03-21 14:00:00
12  recommendationservice-2                                 7           2022-03-21 14:00:00         2022-03-21 14:00:00
13            cartservice-0                                 7           2022-03-21 14:00:00         2022-03-21 14:00:00
14            cartservice-2                                 7           2022-03-21 14:00:00         2022-03-21 14:00:00
15            cartservice-1                                 7           2022-03-21 14:00:00         2022-03-21 14:00:00
16  recommendationservice-1                                 7           2022-03-21 14:00:00         2022-03-21 14:00:00
17        shippingservice-1                                 6           2022-03-21 14:00:00         2022-03-21 14:00:00
18        shippingservice-0                                 6           2022-03-21 14:00:00         2022-03-21 14:00:00
19        shippingservice-2                                 6           2022-03-21 14:00:00         2022-03-21 14:00:00,               cmdb_id                                          kpi_name segment_start_timestamp segment_end_timestamp  segment_length_minutes  severity_ratio node_id
0   checkoutservice-2                container.node-5.container_sockets     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.250000  node-5
1   checkoutservice-2       container.node-5.container_file_descriptors     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.100000  node-5
2   checkoutservice-2  container.node-5.container_fs_limit_MB./dev/vda1     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-5
3   checkoutservice-2           container.node-5.container_memory_cache     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-5
4   checkoutservice-2     container.node-5.container_memory_mapped_file     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-5
5   checkoutservice-2    container.node-5.container_memory_max_usage_MB     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-5
6   checkoutservice-2             container.node-5.container_memory_rss     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-5
7   checkoutservice-2        container.node-5.container_spec_cpu_period     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-5
8   checkoutservice-2         container.node-5.container_spec_cpu_quota     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-5
9   checkoutservice-2        container.node-5.container_spec_cpu_shares     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-5
10         frontend-0        container.node-6.container_memory_usage_MB     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000903  node-6
11  checkoutservice-0       container.node-6.container_file_descriptors     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6
12  checkoutservice-0  container.node-6.container_fs_limit_MB./dev/vda1     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6
13  checkoutservice-0     container.node-6.container_memory_mapped_file     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6
14  checkoutservice-0    container.node-6.container_memory_max_usage_MB     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6
15  checkoutservice-0             container.node-6.container_memory_rss     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6
16  checkoutservice-0                container.node-6.container_sockets     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6
17  checkoutservice-0        container.node-6.container_spec_cpu_period     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6
18  checkoutservice-0         container.node-6.container_spec_cpu_quota     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6
19  checkoutservice-0        container.node-6.container_spec_cpu_shares     2022-03-21 14:00:00   2022-03-21 14:30:00                      31        1.000000  node-6)```
```